<a href="https://colab.research.google.com/github/simionRiccard0/Deep-Learning-Project/blob/modifications/viral_genomes_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DNA sequences for identifying viral
genomes in human samples (la falda)

In [1]:
!git clone https://github.com/NeuroCSUT/ViraMiner.git

Cloning into 'ViraMiner'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 78 (delta 10), reused 10 (delta 10), pack-reused 63 (from 1)
Receiving objects: 100% (78/78), 247.04 MiB | 21.49 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Updating files: 100% (57/57), done.


In [2]:
import os
os.listdir('ViraMiner/data/DNA_data/')

['exp11_2014_G7.csv',
 'exp8_2014_G1.csv',
 'exp6_2014_E1.csv',
 'exp2_2013_E2_SCC.csv',
 'exp5_2014_D3.csv',
 'create_LOO_set.py',
 'exp14_2014_O.csv',
 'exp15_2014_P.csv',
 'create_dataset.py',
 'exp17_2015_F2.csv',
 'fullset_train.csv',
 'exp18_2015_F.csv',
 'exp13_2014_N1.csv',
 'exp0_2011_G5.csv',
 'exp3_2013_H4.csv',
 'fullset_validation.csv',
 'exp4_2014_B.csv',
 'exp7_2014_F1.csv',
 'exp9_2014_G5.csv',
 'exp10_2014_G6.csv',
 'fullset_test.csv',
 'exp12_2014_J1.csv',
 'exp1_2011_N19.csv',
 'exp16_2014_R1.csv']

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [4]:
import pandas as pd

train = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_train.csv',
    header=None
)

val = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_validation.csv',
    header=None
)

test = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_test.csv',
    header=None
)

print("Train shape:", train.shape)
print(train.head())

Train shape: (211239, 3)
                                                 0  \
0  seq1006848_2014_P_Lagheden_Matti-HPV-vacc-CIN31   
1                     seq4116690_2014_G6_Hultin_MS   
2            seq360982_2014_F1_PARAFFINBLANKBLOCKS   
3             seq1135576_2011_N19_VIRASKINFAPMISEQ   
4                        seq277_2014_G6_Hultin_MS8   

                                                   1  2  
0  CAAGCCAAGATTTTCTCGCGTCACACTACTCATGACCATTGTATTA...  0  
1  AACGAAGCACGGGCCGAGAGATTGAGGAACCAAGGTCCAGCTCTAG...  0  
2  TAGTGGGTGAGGTTTCTATTTCCATAATGATCTCGCCTCAATTACT...  0  
3  ATATGACCATTCTTGCAAGGTAACACAGGTACATTTTCACAAAGTG...  0  
4  GGTCTTAAAACAACAGAAATTTTTTCCATCACAGTTGCAGAAATTA...  0  


In [5]:
print("Sequence length:", len(train[1][0]))

print("\nClass balance:")
print(train[2].value_counts())

Sequence length: 300

Class balance:
2
0    206773
1      4466
Name: count, dtype: int64


DNA sequences encoding:

In [6]:
import torch
from torch.utils.data import Dataset
#DNA one-hot encoding and reverse complement augmentation
mapping = {
    "A": [1,0,0,0],
    "C": [0,1,0,0],
    "G": [0,0,1,0],
    "T": [0,0,0,1],
    "N": [0,0,0,0]
}
# Reverse complement mapping
COMPLEMENT = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
MAX_LEN = 300
# Function to get reverse complement of a DNA sequence
def reverse_complement(seq):
    return "".join(COMPLEMENT.get(b, "N") for b in reversed(seq))

def encode_sequence(seq):
    seq = seq[:MAX_LEN]
    encoded = [mapping.get(base, [0,0,0,0]) for base in seq]
    while len(encoded) < MAX_LEN:
        encoded.append([0,0,0,0])
    return encoded


class DNADataset(Dataset):
    def __init__(self, dataframe, augment=False):  # <- new parameter
        self.sequences = dataframe[1].values
        self.labels    = dataframe[2].values
        self.augment   = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]

        # Augmentation: 50% probability of using reverse compliment
        if self.augment and torch.rand(1).item() < 0.5:
            seq = reverse_complement(seq)

        x = torch.tensor(encode_sequence(seq), dtype=torch.float32)
        y = torch.tensor(label, dtype=torch.float32)
        return x, y

Preparing the dataset (DataLoader):

In [7]:
from torch.utils.data import DataLoader
use_pin = torch.cuda.is_available()  # True only if there's GPU available


#Add reverse complement augmentation to training dataset
train_dataset = DNADataset(train, augment=True)
val_dataset   = DNADataset(val, augment=False)
test_dataset  = DNADataset(test, augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=use_pin
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=use_pin
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=use_pin
)

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 300, 4])
torch.Size([128])


CNN Branches

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class PatternBranch(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.conv6  = nn.Conv1d(4, 512, 6,  padding='same')
        self.conv12 = nn.Conv1d(4, 512, 12, padding='same')
        self.conv18 = nn.Conv1d(4, 512, 18, padding='same')
        self.conv24 = nn.Conv1d(4, 512, 24, padding='same')

        total = 512 * 4
        self.bn   = nn.BatchNorm1d(total)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(total, 512),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        branches = [
            F.gelu(self.conv6(x)),
            F.gelu(self.conv12(x)),
            F.gelu(self.conv18(x)),
            F.gelu(self.conv24(x)),
        ]
        x = torch.cat(branches, dim=1)
        x = self.bn(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

In [9]:
class FrequencyBranch(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.conv6  = nn.Conv1d(4, 256, 6,  padding='same')
        self.conv12 = nn.Conv1d(4, 256, 12, padding='same')
        self.conv18 = nn.Conv1d(4, 256, 18, padding='same')
        self.conv24 = nn.Conv1d(4, 256, 24, padding='same')

        total = 256 * 4
        self.bn   = nn.BatchNorm1d(total)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(total, 512),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        branches = [
            F.gelu(self.conv6(x)),
            F.gelu(self.conv12(x)),
            F.gelu(self.conv18(x)),
            F.gelu(self.conv24(x)),
        ]
        x = torch.cat(branches, dim=1)
        x = self.bn(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

Transformer branch

In [10]:
class TransformerBranch(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=4, max_len=300, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(4, d_model)
        self.pos_enc   = nn.Embedding(max_len + 1, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.out  = nn.Linear(d_model, 512)

    def forward(self, x):
        B, L, _ = x.shape
        x = self.embedding(x)
        positions = torch.arange(1, L + 1, device=x.device).unsqueeze(0)
        x = x + self.pos_enc(positions)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x = self.transformer(x)
        x = self.norm(x)
        return self.out(x[:, 0, :])

ViraExplorer Model (CNN Pattern branch + CNN Frequency branch + Transformer)

In [11]:
class ViraExplorer(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.pattern     = PatternBranch(dropout)
        self.frequency   = FrequencyBranch(dropout)
        self.transformer = TransformerBranch()
        self.fc = nn.Sequential(
            nn.Linear(1536, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(512),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        p = self.pattern(x)
        f = self.frequency(x)
        t = self.transformer(x)
        x = torch.cat([p, f, t], dim=1)
        return self.fc(x)

Focal Loss

In [12]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.unsqueeze(1)
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt           = torch.exp(-bce)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = ViraExplorer(dropout=0.4).to(device)
criterion = FocalLoss(alpha=0.75, gamma=2.0)

transformer_params = list(model.transformer.parameters())
other_params = list(model.pattern.parameters()) + list(model.frequency.parameters()) + list(model.fc.parameters())

optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": 1e-4},
    {"params": transformer_params, "lr": 3e-5},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

best_auc, patience, counter, EPOCHS = 0, 12, 0, 60

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step()
    model.eval()
    preds, targets_list = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            #Original prediction line
            probs_orig = torch.sigmoid(model(X_batch))
            #Reverse complement prediction: reverse the positions' order (dim=1)
            X_batch_rev = torch.flip(X_batch, dims=[1]) # Reverse the sequence order
            X_batch_rev = X_batch_rev[:, :, [3, 2, 1, 0]] #Complement the bases (A<->T, C<->G)
            probs_rev  = torch.sigmoid(model(X_batch_rev))

            #Average the probabilities from original and reverse complement
            probs = (probs_orig + probs_rev) / 2

            preds.extend(probs.cpu().numpy().flatten())
            targets_list.extend(y_batch.numpy())

    auc = roc_auc_score(targets_list, preds)
    print(f"Epoch {epoch+1:3d} | Loss: {total_loss/len(train_loader):.4f} | Val AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        counter = 0
        torch.save(model.state_dict(), "best_viraexplorer_v2.pth")
    else:
        counter += 1
        if counter >= patience: break

Device: cuda


/tmp/ipykernel_9630/350706705.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


Epoch   1 | Loss: 0.0239 | Val AUC: 0.5754
Epoch   2 | Loss: 0.0221 | Val AUC: 0.8152
Epoch   3 | Loss: 0.0190 | Val AUC: 0.8360
Epoch   4 | Loss: 0.0185 | Val AUC: 0.8423
Epoch   5 | Loss: 0.0182 | Val AUC: 0.8496
Epoch   6 | Loss: 0.0177 | Val AUC: 0.8591
Epoch   7 | Loss: 0.0171 | Val AUC: 0.8679
Epoch   8 | Loss: 0.0168 | Val AUC: 0.8732
Epoch   9 | Loss: 0.0166 | Val AUC: 0.8736
Epoch  10 | Loss: 0.0165 | Val AUC: 0.8767
Epoch  11 | Loss: 0.0162 | Val AUC: 0.8805
Epoch  12 | Loss: 0.0161 | Val AUC: 0.8813
Epoch  13 | Loss: 0.0160 | Val AUC: 0.8839
Epoch  14 | Loss: 0.0157 | Val AUC: 0.8868
Epoch  15 | Loss: 0.0155 | Val AUC: 0.8889
Epoch  16 | Loss: 0.0153 | Val AUC: 0.8895
Epoch  17 | Loss: 0.0152 | Val AUC: 0.8939
Epoch  18 | Loss: 0.0150 | Val AUC: 0.8951
Epoch  19 | Loss: 0.0149 | Val AUC: 0.8969
Epoch  20 | Loss: 0.0148 | Val AUC: 0.8986
Epoch  21 | Loss: 0.0146 | Val AUC: 0.9011
Epoch  22 | Loss: 0.0145 | Val AUC: 0.9010
Epoch  23 | Loss: 0.0144 | Val AUC: 0.9014
Epoch  24 |